# Toy-model sweep on Kaggle (now set to the Stage 2 pilot, grid G-S2P)

Runs this account's share of a sweep grid in the background on Kaggle's GPU and writes a small results file. Built for **two accounts** working on one grid: both use the same `SPLIT = "1,1"`; account 1 runs `PARTS = [0]`, account 2 runs `PARTS = [1]`. The split depends only on the grid, so the accounts never need to talk to each other.

**One-time setup per account**
1. Create a new notebook and import this file (File > Import notebook).
2. Right sidebar > Session options: **Accelerator = GPU P100** (or T4 x2), **Internet = On** (Kaggle may ask you to verify your phone number first).
3. Fill in `COMMIT` (and `PARTS` for the account) in the settings cell.

**Run it in the background:** click **Save Version** > **Save & Run All (Commit)** > Save. Kaggle runs the whole notebook on its servers; you can close the browser. Open the notebook's **Output** tab later to download the zip.

**If you pick T4 x2 with a single account**, set `PARTS = [0, 1]`: each part runs on its own GPU at the same time.

The code is pinned to the commit, so every machine runs identical code. Training is full float32.

In [ ]:
# ---- settings: edit these ----
REPO_URL = "https://github.com/adi5022/Improving-Parameter-Efficiency-in-Transformer-Language-Models-Through-Superposition-Guided-Training.git"
COMMIT   = ""          # full commit hash that contains the grid below (`git rev-parse HEAD` after you commit and push)
GRID     = "G-S2P"      # Stage 2 pilot: 54 runs (oracle-label leakage penalty, then SAE-label leakage penalty at width 63). Earlier grids: G-E, G-D
SPLIT    = "1,1"       # two equal parts
PARTS    = [0, 1]      # T4 x2 on one account: both parts at once. Two accounts instead: [0] on one and [1] on the other
# If the repository is private, add a GitHub token under Add-ons > Secrets named GH_TOKEN.
assert COMMIT, "Fill in COMMIT first"

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Session options > Accelerator"
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
import os
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass                      # no token: fine for a public repository
os.chdir("/kaggle/working")
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", url, "repo"], check=True)
os.chdir("/kaggle/working/repo")
subprocess.run(["git", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("uncommitted in toy/configs:", subprocess.run(["git", "status", "--short", "toy", "configs"], capture_output=True, text=True).stdout or "none")

In [ ]:
# What this account will run (trains nothing).
for part in PARTS:
    print(subprocess.run(["python", "-m", "toy.sweep", "--grid", GRID, "--split", SPLIT, "--part", str(part), "--dry-run"],
                         capture_output=True, text=True).stdout.splitlines()[-1][:60])

In [ ]:
# Run it. Each part is its own process; with two GPUs and two parts, each gets one GPU.
import time
n_gpu = torch.cuda.device_count()
procs = []
for i, part in enumerate(PARTS):
    env = dict(os.environ)
    if len(PARTS) > 1 and n_gpu >= len(PARTS):
        env["CUDA_VISIBLE_DEVICES"] = str(i)
    log = open(f"/kaggle/working/sweep_part{part}.log", "w")
    p = subprocess.Popen(["python", "-u", "-m", "toy.sweep", "--grid", GRID, "--split", SPLIT, "--part", str(part)],
                         stdout=log, stderr=subprocess.STDOUT, env=env)
    procs.append((part, p, f"/kaggle/working/sweep_part{part}.log"))
seen = {part: 0 for part, _, _ in procs}
while any(p.poll() is None for _, p, _ in procs):
    for part, _, path in procs:
        lines = open(path).read().splitlines()
        for l in lines[seen[part]:]:
            print(f"[part {part}] {l}", flush=True)
        seen[part] = len(lines)
    time.sleep(20)
for part, p, path in procs:
    lines = open(path).read().splitlines()
    for l in lines[seen[part]:]:
        print(f"[part {part}] {l}")
    print(f"part {part} exit code {p.returncode}")
assert all(p.returncode == 0 for _, p, _ in procs), "a part failed: see the log above"

## Results file

The next cell prints a table and writes the files into `/kaggle/working` (they appear in the notebook's **Output** tab):

- `<grid>_parts<N>_small.zip`: run summaries and a report (a few KB). **This is the one to hand over.**
- `<grid>_parts<N>_full.zip`: every run including the learned weights (tens of MB), only needed for weight plots and Stage 2 on frozen models.
- the `repo/results` folder also still holds the same files unzipped.

Download both zips from the Output tab and save them into the project under `results/colab/`.

In [ ]:
import json, shutil, collections, statistics, platform, datetime
out = f"/kaggle/working/repo/results/stage0/{GRID}"
rows = [json.loads(l) for l in open(f"{out}/summary.jsonl") if l.strip()]
mine = {r["run_id"]: r for r in rows}
by = collections.defaultdict(list)
for r in mine.values():
    by[(r["width"], r.get("penalty", "none"), r.get("sae_type", "") if r.get("penalty") == "leak_sae" else "", r.get("lam", 0.0))].append(r)
print("mean over seeds: test error | true overlap (mso_all) | seeds | not-ok runs")
for key in sorted(by):
    rs = by[key]
    w, pen, sae, lam = key
    ok = [r for r in rs if r["status"] == "ok"]
    if ok:
        print(f"m={w:>3} {pen:11s} {sae:9s} lam={lam:<7g} | {statistics.mean(r['test_loss'] for r in ok):.3e} | "
              f"{statistics.mean(r['mso_all'] for r in ok):.4f} | {len(ok)} | {len(rs) - len(ok)}")
    else:
        print(f"m={w:>3} {pen:11s} {sae:9s} lam={lam:<7g} | all runs failed")
report = {
    "grid": GRID, "split": SPLIT, "parts": PARTS, "commit": COMMIT, "precision": "fp32",
    "gpu": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "torch": torch.__version__, "python": platform.python_version(),
    "finished_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "runs_done": len(mine), "runs_not_ok": [r["run_id"] for r in mine.values() if r["status"] != "ok"],
    "total_group_minutes": round(sum(r["train_seconds"] for r in mine.values()) / 60, 1),
}
print(json.dumps(report, indent=1))
json.dump(report, open(f"{out}/report.json", "w"), indent=2)
os.makedirs("/kaggle/working/small", exist_ok=True)
shutil.copy(f"{out}/summary.jsonl", "/kaggle/working/small/summary.jsonl")
shutil.copy(f"{out}/report.json", "/kaggle/working/small/report.json")
tag = f"{GRID}_parts{''.join(map(str, PARTS))}"
shutil.make_archive(f"/kaggle/working/{tag}_small", "zip", "/kaggle/working/small")
print("wrote", f"/kaggle/working/{tag}_small.zip")
# every run including the learned weights, as one zip (tens of MB), so it shows up as a single file in the Output tab
shutil.make_archive(f"/kaggle/working/{tag}_full", "zip", out)
print("wrote", f"/kaggle/working/{tag}_full.zip", round(os.path.getsize(f"/kaggle/working/{tag}_full.zip") / 1e6, 1), "MB")